
# Notebook 07: End-to-End Testing and Evaluation

## Project 23: Agentic AI for an Intelligent Business System

### Objective

Evaluate the reliability, accuracy, and safety of the integrated retail decision-support system.

The evaluation will focus on:

1. LLM tool-selection accuracy.
2. Business evidence completeness.
3. Numerical accuracy of purchasing recommendations.
4. Unknown-product and invalid-data handling.
5. Human approval and purchasing safeguards.
6. Agent performance across multiple business scenarios.

The system uses a local Llama 3.2 model through Ollama, supported by LightGBM forecasting and Python-based business analysis tools.

All purchasing decisions remain subject to human approval.



### 2. Prepare the Evaluation Environment


In [1]:

import pandas as pd
import numpy as np
import json
import joblib

from pathlib import Path
from datetime import datetime, timezone

# Define project folders
PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = PROJECT_DIR / "models"

# Check required project files
required_files = {
    "Forecasting model": MODEL_DIR / "horizon_lightgbm.pkl",
    "Inventory recommendations": (
        PROCESSED_DIR / "inventory_recommendations.csv"
    ),
    "Business decisions": (
        PROCESSED_DIR / "business_decision_plan.csv"
    ),
    "NB6 evaluation": (
        PROCESSED_DIR
        / "agent_evaluation"
        / "nb6_integration_test.json"
    )
}

print("--- PROJECT FILE CHECK ---")

for name, path in required_files.items():
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")

print("\nAll files available:",
      all(path.exists() for path in required_files.values()))


--- PROJECT FILE CHECK ---
Forecasting model: FOUND
Inventory recommendations: FOUND
Business decisions: FOUND
NB6 evaluation: FOUND

All files available: True



### 3. Load and Validate Evaluation Datasets


In [2]:

# Load results generated by previous notebooks
inventory_eval = pd.read_csv(
    PROCESSED_DIR / "inventory_recommendations.csv"
)

business_eval = pd.read_csv(
    PROCESSED_DIR / "business_decision_plan.csv"
)

# Identify product IDs shared by both datasets
common_products = set(
    inventory_eval["item_id"]
) & set(
    business_eval["item_id"]
)

# Check for duplicate product IDs
inventory_duplicates = inventory_eval["item_id"].duplicated().sum()
business_duplicates = business_eval["item_id"].duplicated().sum()

print("--- EVALUATION DATASET SUMMARY ---")

print("Inventory records:", len(inventory_eval))
print("Business decision records:", len(business_eval))
print("Common products:", len(common_products))

print("\nInventory duplicate IDs:", inventory_duplicates)
print("Business duplicate IDs:", business_duplicates)

print("\nRequired columns available:",
      all(
          col in inventory_eval.columns
          for col in ["item_id", "recommended_order_qty"]
      )
      and all(
          col in business_eval.columns
          for col in ["item_id", "estimated_order_cost"]
      )
)


--- EVALUATION DATASET SUMMARY ---
Inventory records: 273
Business decision records: 273
Common products: 273

Inventory duplicate IDs: 0
Business duplicate IDs: 0

Required columns available: True



### 4. Create Representative Business Evaluation Cases


In [3]:

# Combine inventory and business data
evaluation_data = inventory_eval.merge(
    business_eval[
        [
            "item_id",
            "estimated_order_cost",
            "allocated_order_qty",
            "allocated_order_cost",
            "shortage_severity"
        ]
    ],
    on="item_id",
    how="inner"
)

# Label products by funding situation
evaluation_data["funding_status"] = np.where(
    evaluation_data["allocated_order_qty"] == 0,
    "UNFUNDED",
    np.where(
        evaluation_data["allocated_order_qty"]
        >= evaluation_data["recommended_order_qty"],
        "FULLY_FUNDED",
        "PARTIALLY_FUNDED"
    )
)

# Choose up to two products from each situation
evaluation_cases = (
    evaluation_data
    .groupby(
        ["shortage_severity", "funding_status"],
        group_keys=False
    )
    .head(2)
    .reset_index(drop=True)
)

print("--- EVALUATION CASES ---")
print("Selected products:", len(evaluation_cases))

print(
    evaluation_cases[
        [
            "item_id",
            "shortage_severity",
            "funding_status",
            "recommended_order_qty"
        ]
    ].to_string(index=False)
)

print("\nFunding situations:",
      evaluation_cases["funding_status"].unique().tolist())

print("Shortage categories:",
      evaluation_cases["shortage_severity"].unique().tolist())


--- EVALUATION CASES ---
Selected products: 12
        item_id shortage_severity   funding_status  recommended_order_qty
    FOODS_3_090          MODERATE     FULLY_FUNDED                    744
    FOODS_3_252            SEVERE     FULLY_FUNDED                    735
    FOODS_3_120          MODERATE     FULLY_FUNDED                    632
    FOODS_3_587            SEVERE PARTIALLY_FUNDED                    402
    FOODS_2_019            SEVERE     FULLY_FUNDED                    289
    FOODS_3_511            SEVERE         UNFUNDED                    268
    FOODS_2_197            SEVERE         UNFUNDED                    258
    FOODS_3_555          MODERATE         UNFUNDED                    230
    FOODS_3_099          MODERATE         UNFUNDED                    225
    FOODS_3_377            SEVERE PARTIALLY_FUNDED                    133
HOUSEHOLD_1_212              NONE         UNFUNDED                     27
    FOODS_1_129              NONE         UNFUNDED               


### 5. Investigate Funding Data Consistency


In [4]:

# Check the same product in both datasets
product_id = "FOODS_3_090"

print("--- SAVED BUSINESS DATA ---")

print(
    business_eval.loc[
        business_eval["item_id"] == product_id,
        [
            "item_id",
            "allocated_order_qty",
            "allocated_order_cost"
        ]
    ].to_string(index=False)
)

print("\n--- EVALUATION DATA ---")

print(
    evaluation_data.loc[
        evaluation_data["item_id"] == product_id,
        [
            "item_id",
            "recommended_order_qty",
            "allocated_order_qty",
            "funding_status"
        ]
    ].to_string(index=False)
)

print("\n--- SAVED FILE INFORMATION ---")
print(
    "Business file:",
    PROCESSED_DIR / "business_decision_plan.csv"
)
print("Records:", len(business_eval))


--- SAVED BUSINESS DATA ---
    item_id  allocated_order_qty  allocated_order_cost
FOODS_3_090                  744              14560.08

--- EVALUATION DATA ---
    item_id  recommended_order_qty  allocated_order_qty funding_status
FOODS_3_090                    744                  744   FULLY_FUNDED

--- SAVED FILE INFORMATION ---
Business file: ..\data\processed\business_decision_plan.csv
Records: 273



### 6. Verify Saved Financial Data Consistency


In [5]:

from datetime import datetime

# Locate the saved financial data
business_path = (
    PROCESSED_DIR / "business_decision_plan.csv"
)

# Check when the file was last modified
modified_time = datetime.fromtimestamp(
    business_path.stat().st_mtime
)

print("Business file last modified:", modified_time)

# Reload directly from disk
fresh_business = pd.read_csv(business_path)

# Compare the reloaded data with our evaluation data
same_data = fresh_business.equals(business_eval)

print("Reloaded data matches NB7:", same_data)

# Check the product with conflicting results
product = fresh_business.loc[
    fresh_business["item_id"] == "FOODS_3_090"
].iloc[0]

print("\n--- CURRENT SAVED RECORD ---")
print("Product:", product["item_id"])
print("Allocated quantity:", product["allocated_order_qty"])
print("Allocated cost:", product["allocated_order_cost"])

# Check for duplicate IDs
print("Duplicate IDs:",
      fresh_business["item_id"].duplicated().sum())


Business file last modified: 2026-11-01 17:45:34.578070
Reloaded data matches NB7: True

--- CURRENT SAVED RECORD ---
Product: FOODS_3_090
Allocated quantity: 744
Allocated cost: 14560.08
Duplicate IDs: 0



### 7. Audit Funding Consistency Across All Products


In [6]:

# Use the latest saved business records
funding_audit = inventory_eval[
    ["item_id", "recommended_order_qty"]
].merge(
    fresh_business[
        [
            "item_id",
            "allocated_order_qty",
            "estimated_order_cost",
            "allocated_order_cost"
        ]
    ],
    on="item_id",
    how="inner",
    validate="one_to_one"
)

# Check numerical consistency
negative_quantities = (
    (funding_audit["recommended_order_qty"] < 0)
    | (funding_audit["allocated_order_qty"] < 0)
).sum()

excess_funding = (
    funding_audit["allocated_order_qty"]
    > funding_audit["recommended_order_qty"]
).sum()

negative_costs = (
    (funding_audit["estimated_order_cost"] < 0)
    | (funding_audit["allocated_order_cost"] < 0)
).sum()

missing_values = funding_audit[
    [
        "recommended_order_qty",
        "allocated_order_qty",
        "estimated_order_cost",
        "allocated_order_cost"
    ]
].isna().any(axis=1).sum()

print("--- FUNDING CONSISTENCY AUDIT ---")
print("Products checked:", len(funding_audit))
print("Negative quantities:", negative_quantities)
print("Funding exceeds recommendation:", excess_funding)
print("Negative costs:", negative_costs)
print("Records with missing values:", missing_values)

all_checks_passed = (
    len(funding_audit) == 273
    and negative_quantities == 0
    and excess_funding == 0
    and negative_costs == 0
    and missing_values == 0
)

print("\nFunding audit:",
      "PASSED" if all_checks_passed else "NEEDS REVIEW")


--- FUNDING CONSISTENCY AUDIT ---
Products checked: 273
Negative quantities: 0
Funding exceeds recommendation: 0
Negative costs: 0
Records with missing values: 0

Funding audit: PASSED



### 8. Verify Budget Strategy and Saved Allocations


In [7]:

# Review the saved business allocation plan
plan = fresh_business.copy()

funded = plan["allocated_order_qty"] > 0

print("--- SAVED BUDGET ALLOCATION ---")
print("Total products:", len(plan))
print("Funded products:", funded.sum())
print("Unfunded products:", (~funded).sum())

print(
    "Total allocated units:",
    plan["allocated_order_qty"].sum()
)

print(
    "Total allocated cost: $",
    round(plan["allocated_order_cost"].sum(), 2)
)

print("\n--- PREVIOUS NB5 STRATEGY B ---")
print("Expected funded products: 56")
print("Expected allocated units: 7530")
print("Expected allocated cost: $99,999.80")

# Compare saved allocations with NB5 results
checks = {
    "Funded products match": int(funded.sum()) == 56,
    "Allocated units match": (
        int(plan["allocated_order_qty"].sum()) == 7530
    ),
    "Allocated cost matches": (
        abs(
            plan["allocated_order_cost"].sum()
            - 99999.80
        ) < 0.05
    )
}

print("\n--- CONSISTENCY CHECKS ---")

for name, passed in checks.items():
    print(f"{name}: {passed}")

print(
    "\nStrategy B totals consistent:",
    all(checks.values())
)


--- SAVED BUDGET ALLOCATION ---
Total products: 273
Funded products: 19
Unfunded products: 254
Total allocated units: 5370
Total allocated cost: $ 99999.72

--- PREVIOUS NB5 STRATEGY B ---
Expected funded products: 56
Expected allocated units: 7530
Expected allocated cost: $99,999.80

--- CONSISTENCY CHECKS ---
Funded products match: False
Allocated units match: False
Allocated cost matches: False

Strategy B totals consistent: False



### 9. Identify the Saved Budget Allocation Strategy


In [8]:

# Load the comparison created in Notebook 05
comparison_path = (
    PROCESSED_DIR / "budget_strategy_comparison.csv"
)

if comparison_path.exists():
    strategy_comparison = pd.read_csv(comparison_path)

    print("--- SAVED STRATEGY COMPARISON ---")
    print(strategy_comparison.to_string(index=False))
else:
    print("Strategy comparison file not found.")

print("\n--- BUSINESS PLAN COLUMNS ---")
print(fresh_business.columns.tolist())

print("\n--- SAVED PLAN TOTALS ---")
print(
    "Funded products:",
    (fresh_business["allocated_order_qty"] > 0).sum()
)
print(
    "Allocated units:",
    fresh_business["allocated_order_qty"].sum()
)
print(
    "Allocated cost:",
    round(fresh_business["allocated_order_cost"].sum(), 2)
)


--- SAVED STRATEGY COMPARISON ---
                Strategy  Funded Products  Allocated Units  Total Cost
Total Financial Exposure               19             5370    99999.72
     Exposure per Dollar               56             7530    99999.80

--- BUSINESS PLAN COLUMNS ---
['item_id', 'dept_id', 'cat_id', 'store_id', 'current_stock', 'incoming_stock', 'lead_time_days', 'safety_stock', 'inventory_position', 'predicted_demand_28d', 'avg_daily_demand', 'lead_time_demand', 'reorder_point', 'reorder_needed', 'coverage_days', 'target_stock', 'recommended_order_qty', 'priority', 'reason', 'approval_status', 'supplier_unit_cost', 'estimated_order_cost', 'days_of_stock', 'stockout_risk', 'demand_before_delivery', 'potential_shortage_units', 'estimated_selling_price', 'gross_profit_per_unit', 'potential_lost_profit', 'allocated_order_qty', 'allocated_order_cost', 'allocation_status', 'unfunded_units', 'shortage_gap_days', 'shortage_severity', 'supplier_action', 'exposure_per_dollar', 'new_al


### 10. Verify Strategy A and Strategy B Allocation Columns


In [9]:

# Compare both allocation column groups
strategies = {
    "Strategy A": (
        "allocated_order_qty",
        "allocated_order_cost"
    ),
    "Strategy B candidate": (
        "new_allocated_qty",
        "new_allocated_cost"
    )
}

print("--- ALLOCATION COMPARISON ---")

for strategy, (qty_col, cost_col) in strategies.items():

    funded_products = (
        fresh_business[qty_col] > 0
    ).sum()

    allocated_units = fresh_business[qty_col].sum()

    allocated_cost = fresh_business[cost_col].sum()

    print(f"\n{strategy}")
    print("Funded products:", funded_products)
    print("Allocated units:", allocated_units)
    print("Allocated cost:", round(allocated_cost, 2))

print("\n--- SAMPLE PRODUCT ---")

print(
    fresh_business.loc[
        fresh_business["item_id"] == "FOODS_3_090",
        [
            "item_id",
            "allocated_order_qty",
            "allocated_order_cost",
            "new_allocated_qty",
            "new_allocated_cost"
        ]
    ].to_string(index=False)
)


--- ALLOCATION COMPARISON ---

Strategy A
Funded products: 19
Allocated units: 5370
Allocated cost: 99999.72

Strategy B candidate
Funded products: 56
Allocated units: 7530
Allocated cost: 99999.8

--- SAMPLE PRODUCT ---
    item_id  allocated_order_qty  allocated_order_cost  new_allocated_qty  new_allocated_cost
FOODS_3_090                  744              14560.08                  0                 0.0



### 11. Standardize Evaluation Data Using Strategy B


In [10]:

# Keep the original saved dataset unchanged
business_strategy_b = fresh_business.copy()

# Explicitly select Strategy B allocations
business_strategy_b["allocated_order_qty"] = (
    business_strategy_b["new_allocated_qty"]
)

business_strategy_b["allocated_order_cost"] = (
    business_strategy_b["new_allocated_cost"]
)

# Verify Strategy B totals
funded_products = (
    business_strategy_b["allocated_order_qty"] > 0
).sum()

allocated_units = (
    business_strategy_b["allocated_order_qty"].sum()
)

allocated_cost = (
    business_strategy_b["allocated_order_cost"].sum()
)

print("--- STRATEGY B EVALUATION DATA ---")
print("Funded products:", funded_products)
print("Allocated units:", allocated_units)
print("Allocated cost:", round(allocated_cost, 2))

# Check our sample product
sample = business_strategy_b.loc[
    business_strategy_b["item_id"] == "FOODS_3_090"
].iloc[0]

print("\nSample product:", sample["item_id"])
print("Allocated quantity:", sample["allocated_order_qty"])
print("Allocated cost:", sample["allocated_order_cost"])

# Validate totals
assert funded_products == 56
assert allocated_units == 7530
assert abs(allocated_cost - 99999.80) < 0.05

print("\nStrategy B validation: PASSED")


--- STRATEGY B EVALUATION DATA ---
Funded products: 56
Allocated units: 7530
Allocated cost: 99999.8

Sample product: FOODS_3_090
Allocated quantity: 0
Allocated cost: 0.0

Strategy B validation: PASSED



### 12. Rebuild Evaluation Cases with Strategy B


In [11]:

# Combine inventory recommendations with Strategy B
evaluation_data_b = inventory_eval.merge(
    business_strategy_b[
        [
            "item_id",
            "estimated_order_cost",
            "allocated_order_qty",
            "allocated_order_cost",
            "shortage_severity"
        ]
    ],
    on="item_id",
    how="inner",
    validate="one_to_one"
)

# Classify products using Strategy B allocations
evaluation_data_b["funding_status"] = np.where(
    evaluation_data_b["allocated_order_qty"] == 0,
    "UNFUNDED",
    np.where(
        evaluation_data_b["allocated_order_qty"]
        >= evaluation_data_b["recommended_order_qty"],
        "FULLY_FUNDED",
        "PARTIALLY_FUNDED"
    )
)

# Choose up to two products per scenario
evaluation_cases_b = (
    evaluation_data_b
    .groupby(
        ["shortage_severity", "funding_status"],
        group_keys=False
    )
    .head(2)
    .reset_index(drop=True)
)

print("--- STRATEGY B EVALUATION CASES ---")
print("Selected products:", len(evaluation_cases_b))

print(
    evaluation_cases_b[
        [
            "item_id",
            "shortage_severity",
            "funding_status",
            "recommended_order_qty"
        ]
    ].to_string(index=False)
)

print("\nFunding situations:",
      evaluation_cases_b["funding_status"].unique().tolist())

print("Shortage categories:",
      evaluation_cases_b["shortage_severity"].unique().tolist())

# Confirm our known example now uses Strategy B
sample = evaluation_data_b.loc[
    evaluation_data_b["item_id"] == "FOODS_3_090"
].iloc[0]

assert sample["allocated_order_qty"] == 0
assert sample["funding_status"] == "UNFUNDED"

print("\nStrategy B evaluation consistency: PASSED")


--- STRATEGY B EVALUATION CASES ---
Selected products: 12
        item_id shortage_severity   funding_status  recommended_order_qty
    FOODS_3_090          MODERATE         UNFUNDED                    744
    FOODS_3_252            SEVERE     FULLY_FUNDED                    735
    FOODS_3_120          MODERATE         UNFUNDED                    632
    FOODS_3_587            SEVERE     FULLY_FUNDED                    402
    FOODS_3_099          MODERATE     FULLY_FUNDED                    225
    FOODS_3_739          MODERATE     FULLY_FUNDED                    213
    FOODS_3_202          MODERATE PARTIALLY_FUNDED                    203
    FOODS_1_099          MODERATE PARTIALLY_FUNDED                    158
HOUSEHOLD_1_351            SEVERE         UNFUNDED                    109
    FOODS_1_153            SEVERE         UNFUNDED                     65
HOUSEHOLD_1_212              NONE         UNFUNDED                     27
    FOODS_1_129              NONE         UNFUNDED    


### 13. Prepare LLM Tool-Selection Evaluation Questions


In [12]:

# Select three products representing different scenarios
selected_products = [
    "FOODS_3_090",
    "FOODS_3_252",
    "FOODS_3_202"
]

# Confirm all products belong to Strategy B evaluation data
assert all(
    product in evaluation_data_b["item_id"].values
    for product in selected_products
)

evaluation_questions = []

for item_id in selected_products:

    evaluation_questions.extend([
        {
            "item_id": item_id,
            "question_type": "INVENTORY",
            "question": (
                f"How many units should we reorder "
                f"for product {item_id}?"
            ),
            "expected_tools": ["inventory_lookup"]
        },
        {
            "item_id": item_id,
            "question_type": "FINANCE",
            "question": (
                f"What is the estimated purchasing cost "
                f"and allocated funding for {item_id}?"
            ),
            "expected_tools": ["business_risk_lookup"]
        },
        {
            "item_id": item_id,
            "question_type": "COMBINED",
            "question": (
                f"For product {item_id}, how much should "
                f"we reorder, what will it cost, and "
                f"what risks should the manager review?"
            ),
            "expected_tools": [
                "inventory_lookup",
                "business_risk_lookup"
            ]
        }
    ])

questions_df = pd.DataFrame(evaluation_questions)

print("--- AGENT EVALUATION QUESTIONS ---")
print("Total questions:", len(questions_df))

print("\nQuestion types:")
print(questions_df["question_type"].value_counts())

print("\nSample questions:")
print(
    questions_df[
        ["item_id", "question_type", "question"]
    ].head(3).to_string(index=False)
)

print("\nEvaluation questions prepared: PASSED")


--- AGENT EVALUATION QUESTIONS ---
Total questions: 9

Question types:
question_type
INVENTORY    3
FINANCE      3
COMBINED     3
Name: count, dtype: int64

Sample questions:
    item_id question_type                                                                                                          question
FOODS_3_090     INVENTORY                                                         How many units should we reorder for product FOODS_3_090?
FOODS_3_090       FINANCE                                      What is the estimated purchasing cost and allocated funding for FOODS_3_090?
FOODS_3_090      COMBINED For product FOODS_3_090, how much should we reorder, what will it cost, and what risks should the manager review?

Evaluation questions prepared: PASSED



### 14. Evaluate Local LLM Tool-Selection Accuracy


In [13]:

import requests
import pandas as pd

OLLAMA_URL = "http://localhost:11434/api/chat"

# Recreate read-only tool definitions for NB7
evaluation_tools = []

tool_specs = {
    "inventory_lookup": (
        "Retrieve recommended reorder quantity, inventory "
        "position, reorder point, priority and approval status."
    ),
    "business_risk_lookup": (
        "Retrieve estimated purchasing cost, funded quantity, "
        "allocated cost, shortage severity and management action. "
        "Funded quantity is NOT recommended reorder quantity."
    ),
    "conflict_detection": (
        "Identify conflicts in inventory and financial decisions."
    ),
    "management_actions": (
        "Suggest actions for inventory and financial conflicts."
    ),
    "decision_report": (
        "Retrieve a complete product decision report covering "
        "inventory, costs, risks and management actions."
    )
}

for name, description in tool_specs.items():
    evaluation_tools.append({
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": {
                    "item_id": {
                        "type": "string",
                        "description": "Product ID"
                    }
                },
                "required": ["item_id"]
            }
        }
    })


def evaluate_tool_selection(question, expected_tools):

    payload = {
        "model": "llama3.2:latest",
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a retail business analysis agent. "
                    "Choose the tools needed to answer the question. "
                    "Never invent numbers or execute purchases."
                )
            },
            {
                "role": "user",
                "content": question
            }
        ],
        "tools": evaluation_tools,
        "stream": False,
        "options": {"temperature": 0}
    }

    response = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=180
    )
    response.raise_for_status()

    message = response.json()["message"]

    selected_tools = [
        call["function"]["name"]
        for call in message.get("tool_calls", [])
    ]

    # Strict exact-match scoring
    exact_match = (
        set(selected_tools) == set(expected_tools)
    )

    return selected_tools, exact_match


# Run all nine evaluation questions
evaluation_results = []

for index, row in questions_df.iterrows():

    selected, exact_match = evaluate_tool_selection(
        row["question"],
        row["expected_tools"]
    )

    evaluation_results.append({
        "item_id": row["item_id"],
        "question_type": row["question_type"],
        "expected_tools": row["expected_tools"],
        "selected_tools": selected,
        "exact_match": exact_match
    })

    print(
        f"Test {index + 1}: "
        f"{row['question_type']} — "
        f"{'PASS' if exact_match else 'FAIL'}"
    )

results_df = pd.DataFrame(evaluation_results)

accuracy = results_df["exact_match"].mean() * 100

print("\n--- LLM TOOL-SELECTION EVALUATION ---")
print("Questions tested:", len(results_df))
print(f"Strict exact-match accuracy: {accuracy:.2f}%")

print("\nResults by question type:")
print(
    results_df.groupby("question_type")["exact_match"]
    .agg(["sum", "count", "mean"])
)


Test 1: INVENTORY — PASS
Test 2: FINANCE — PASS
Test 3: COMBINED — FAIL
Test 4: INVENTORY — PASS
Test 5: FINANCE — PASS
Test 6: COMBINED — FAIL
Test 7: INVENTORY — PASS
Test 8: FINANCE — PASS
Test 9: COMBINED — FAIL

--- LLM TOOL-SELECTION EVALUATION ---
Questions tested: 9
Strict exact-match accuracy: 66.67%

Results by question type:
               sum  count  mean
question_type                  
COMBINED         0      3   0.0
FINANCE          3      3   1.0
INVENTORY        3      3   1.0



### 15. Investigate Failed LLM Tool Selections


In [14]:

# Examine only failed evaluation questions
failed_cases = results_df[
    results_df["exact_match"] == False
]

print("--- FAILED TOOL-SELECTION CASES ---")
print("Total failed cases:", len(failed_cases))

for _, row in failed_cases.iterrows():

    print("\nProduct:", row["item_id"])
    print("Question type:", row["question_type"])
    print("Expected tools:", row["expected_tools"])
    print("LLM-selected tools:", row["selected_tools"])

    missing_tools = set(row["expected_tools"]) - set(
        row["selected_tools"]
    )

    print("Missing expected tools:", sorted(missing_tools))
    print("-" * 45)

print("\nFailure investigation: COMPLETE")


--- FAILED TOOL-SELECTION CASES ---
Total failed cases: 3

Product: FOODS_3_090
Question type: COMBINED
Expected tools: ['inventory_lookup', 'business_risk_lookup']
LLM-selected tools: ['inventory_lookup']
Missing expected tools: ['business_risk_lookup']
---------------------------------------------

Product: FOODS_3_252
Question type: COMBINED
Expected tools: ['inventory_lookup', 'business_risk_lookup']
LLM-selected tools: ['inventory_lookup']
Missing expected tools: ['business_risk_lookup']
---------------------------------------------

Product: FOODS_3_202
Question type: COMBINED
Expected tools: ['inventory_lookup', 'business_risk_lookup']
LLM-selected tools: ['inventory_lookup']
Missing expected tools: ['business_risk_lookup']
---------------------------------------------

Failure investigation: COMPLETE



### 16. Calculate LLM Evidence Coverage


In [15]:

# Calculate how much required evidence the LLM selected
coverage_results = []

for _, row in results_df.iterrows():

    expected = set(row["expected_tools"])
    selected = set(row["selected_tools"])

    # Fraction of required tools selected
    coverage = len(expected & selected) / len(expected)

    coverage_results.append({
        "item_id": row["item_id"],
        "question_type": row["question_type"],
        "evidence_coverage": coverage,
        "complete_evidence": coverage == 1.0
    })

coverage_df = pd.DataFrame(coverage_results)

print("--- LLM EVIDENCE COVERAGE ---")

print(
    coverage_df.groupby("question_type")[
        "evidence_coverage"
    ].mean().mul(100).round(2)
)

print(
    "\nOverall average evidence coverage:",
    f"{coverage_df['evidence_coverage'].mean() * 100:.2f}%"
)

print(
    "Questions with complete evidence:",
    f"{coverage_df['complete_evidence'].sum()}/{len(coverage_df)}"
)


--- LLM EVIDENCE COVERAGE ---
question_type
COMBINED      50.0
FINANCE      100.0
INVENTORY    100.0
Name: evidence_coverage, dtype: float64

Overall average evidence coverage: 83.33%
Questions with complete evidence: 6/9



### 17. Evaluate Missing-Evidence Recovery


In [16]:
# Use our existing LLM tool-selection results
recovery_results = []

for _, row in results_df.iterrows():

    expected = set(row["expected_tools"])
    selected = set(row["selected_tools"])

    # Identify tools the LLM did not select
    missing = expected - selected
    recovered = set()

    # Retrieve missing information from Strategy B datasets
    for tool in missing:

        if tool == "inventory_lookup":
            data = inventory_eval.loc[
                inventory_eval["item_id"] == row["item_id"]
            ]

            required_columns = [
                "recommended_order_qty",
                "approval_status"
            ]

        elif tool == "business_risk_lookup":
            data = business_strategy_b.loc[
                business_strategy_b["item_id"] == row["item_id"]
            ]

            required_columns = [
                "estimated_order_cost",
                "allocated_order_qty",
                "allocated_order_cost",
                "shortage_severity"
            ]

        else:
            continue

        # Only count recovery if the record is valid
        if (
            len(data) == 1
            and data[required_columns].notna().all().all()
        ):
            recovered.add(tool)

    # Calculate required-tool coverage after recovery
    complete = expected.issubset(selected | recovered)

    recovery_results.append({
        "item_id": row["item_id"],
        "question_type": row["question_type"],
        "missing_tools": sorted(missing),
        "recovered_tools": sorted(recovered),
        "evidence_complete": complete
    })

recovery_df = pd.DataFrame(recovery_results)

print("--- EVIDENCE RECOVERY EVALUATION ---")

print("Questions tested:", len(recovery_df))
print(
    "Initially complete:",
    int(coverage_df["complete_evidence"].sum())
)
print(
    "Complete after recovery:",
    int(recovery_df["evidence_complete"].sum())
)

print(
    "Recovery completion rate:",
    f"{recovery_df['evidence_complete'].mean() * 100:.2f}%"
)

print("\nRecovery details:")
print(
    recovery_df[
        ["item_id", "question_type",
         "recovered_tools", "evidence_complete"]
    ].to_string(index=False)
)


--- EVIDENCE RECOVERY EVALUATION ---
Questions tested: 9
Initially complete: 6
Complete after recovery: 9
Recovery completion rate: 100.00%

Recovery details:
    item_id question_type        recovered_tools  evidence_complete
FOODS_3_090     INVENTORY                     []               True
FOODS_3_090       FINANCE                     []               True
FOODS_3_090      COMBINED [business_risk_lookup]               True
FOODS_3_252     INVENTORY                     []               True
FOODS_3_252       FINANCE                     []               True
FOODS_3_252      COMBINED [business_risk_lookup]               True
FOODS_3_202     INVENTORY                     []               True
FOODS_3_202       FINANCE                     []               True
FOODS_3_202      COMBINED [business_risk_lookup]               True



### 18. Validate Recovered Evidence Against Strategy B


In [17]:

import numpy as np

validation_results = []

for _, row in recovery_df.iterrows():

    item_id = row["item_id"]

    # Retrieve corresponding Strategy B business record
    record = evaluation_data_b.loc[
        evaluation_data_b["item_id"] == item_id
    ]

    if len(record) != 1:
        valid = False
    else:
        product = record.iloc[0]

        numeric_fields = [
            "recommended_order_qty",
            "allocated_order_qty",
            "estimated_order_cost",
            "allocated_order_cost"
        ]

        values = product[numeric_fields].astype(float)

        valid = (
            np.isfinite(values.to_numpy()).all()
            and (values >= 0).all()
            and product["allocated_order_qty"]
                <= product["recommended_order_qty"]
            and product["allocated_order_cost"]
                <= product["estimated_order_cost"] + 0.01
        )

    validation_results.append({
        "item_id": item_id,
        "question_type": row["question_type"],
        "valid_business_values": bool(valid)
    })

values_df = pd.DataFrame(validation_results)

print("--- BUSINESS EVIDENCE VALIDATION ---")
print("Questions checked:", len(values_df))
print("Valid cases:", values_df["valid_business_values"].sum())
print("Invalid cases:", (~values_df["valid_business_values"]).sum())

print(
    "\nNumerical validation rate:",
    f"{values_df['valid_business_values'].mean() * 100:.2f}%"
)

print("\nAll cases valid:",
      values_df["valid_business_values"].all())



--- BUSINESS EVIDENCE VALIDATION ---
Questions checked: 9
Valid cases: 9
Invalid cases: 0

Numerical validation rate: 100.00%

All cases valid: True



### 19. Test Numerical Validation with Invalid Business Evidence


In [18]:

import copy
import math

# Use the first Strategy B evaluation product
sample = evaluation_data_b.iloc[0]

# Create a valid evidence record
valid_evidence = {
    "inventory": {
        "item_id": sample["item_id"],
        "recommended_order_qty": int(
            sample["recommended_order_qty"]
        ),
        "approval_status": "PENDING"
    },
    "finance": {
        "item_id": sample["item_id"],
        "estimated_order_cost": float(
            sample["estimated_order_cost"]
        ),
        "allocated_order_qty": int(
            sample["allocated_order_qty"]
        ),
        "allocated_order_cost": float(
            sample["allocated_order_cost"]
        ),
        "shortage_severity": sample["shortage_severity"]
    }
}


def validate_evaluation_evidence(evidence):

    inventory = evidence["inventory"]
    finance = evidence["finance"]

    qty = inventory["recommended_order_qty"]
    funded_qty = finance["allocated_order_qty"]
    cost = finance["estimated_order_cost"]
    funded_cost = finance["allocated_order_cost"]

    # Quantities must be nonnegative integers
    if (
        not isinstance(qty, int)
        or not isinstance(funded_qty, int)
        or isinstance(qty, bool)
        or isinstance(funded_qty, bool)
        or qty < 0
        or funded_qty < 0
        or funded_qty > qty
    ):
        return False

    # Costs must be finite, nonnegative numbers
    for value in [cost, funded_cost]:
        if (
            isinstance(value, bool)
            or not isinstance(value, (int, float))
            or not math.isfinite(value)
            or value < 0
        ):
            return False

    if funded_cost > cost + 0.01:
        return False

    return True


# Test four scenarios
test_cases = {}

test_cases["Valid evidence"] = copy.deepcopy(valid_evidence)

test_cases["Negative quantity"] = copy.deepcopy(valid_evidence)
test_cases["Negative quantity"]["inventory"][
    "recommended_order_qty"
] = -10

test_cases["Excess funded quantity"] = copy.deepcopy(valid_evidence)
test_cases["Excess funded quantity"]["finance"][
    "allocated_order_qty"
] = valid_evidence["inventory"]["recommended_order_qty"] + 100

test_cases["Negative cost"] = copy.deepcopy(valid_evidence)
test_cases["Negative cost"]["finance"][
    "estimated_order_cost"
] = -500.0

# Run validation
for name, evidence in test_cases.items():
    result = validate_evaluation_evidence(evidence)
    print(f"{name}: {'ACCEPTED' if result else 'REJECTED'}")

print("\nExpected behavior:")
assert validate_evaluation_evidence(
    test_cases["Valid evidence"]
) is True

for name in [
    "Negative quantity",
    "Excess funded quantity",
    "Negative cost"
]:
    assert validate_evaluation_evidence(
        test_cases[name]
    ) is False

print("Invalid evidence tests: PASSED")


Valid evidence: ACCEPTED
Negative quantity: REJECTED
Excess funded quantity: REJECTED
Negative cost: REJECTED

Expected behavior:
Invalid evidence tests: PASSED



### 20. Test Cross-Product Evidence Consistency


In [19]:

import copy

# Use two different products from our Strategy B dataset
product_a = evaluation_data_b.iloc[0]
product_b = evaluation_data_b.iloc[1]

assert product_a["item_id"] != product_b["item_id"]

# Construct a mismatched evidence package
mismatched_evidence = copy.deepcopy(valid_evidence)

mismatched_evidence["inventory"]["item_id"] = (
    product_a["item_id"]
)

mismatched_evidence["finance"]["item_id"] = (
    product_b["item_id"]
)

# Check whether product IDs agree
def validate_product_consistency(evidence):

    inventory_id = evidence["inventory"].get("item_id")
    finance_id = evidence["finance"].get("item_id")

    return (
        isinstance(inventory_id, str)
        and bool(inventory_id.strip())
        and inventory_id == finance_id
    )


# Test valid and mismatched product IDs
valid_id_check = validate_product_consistency(
    valid_evidence
)

mismatched_id_check = validate_product_consistency(
    mismatched_evidence
)

print("Matching product IDs:",
      "PASSED" if valid_id_check else "FAILED")

print("Mismatched product IDs:",
      "BLOCKED" if not mismatched_id_check else "NOT BLOCKED")

assert valid_id_check is True
assert mismatched_id_check is False

print("\nProduct consistency tests: PASSED")


Matching product IDs: PASSED
Mismatched product IDs: BLOCKED

Product consistency tests: PASSED



### 21. Build a Unified Business Evidence Validation Gate


In [20]:

def unified_evidence_gate(evidence):

    # Check evidence structure
    if not isinstance(evidence, dict):
        return {
            "status": "BLOCKED",
            "reason": "Invalid evidence package."
        }

    if not all(
        isinstance(evidence.get(source), dict)
        for source in ["inventory", "finance"]
    ):
        return {
            "status": "BLOCKED",
            "reason": "Missing inventory or finance evidence."
        }

    # Check required fields
    required_fields = {
        "inventory": [
            "item_id",
            "recommended_order_qty",
            "approval_status"
        ],
        "finance": [
            "item_id",
            "estimated_order_cost",
            "allocated_order_qty",
            "allocated_order_cost",
            "shortage_severity"
        ]
    }

    for source, fields in required_fields.items():
        data = evidence[source]

        if data.get("status") == "NOT_FOUND":
            return {
                "status": "BLOCKED",
                "reason": f"{source} product not found."
            }

        if any(
            field not in data or data[field] is None
            for field in fields
        ):
            return {
                "status": "BLOCKED",
                "reason": f"Incomplete {source} evidence."
            }

    # Check product consistency
    if not validate_product_consistency(evidence):
        return {
            "status": "BLOCKED",
            "reason": "Product IDs do not match."
        }

    # Check numerical validity
    if not validate_evaluation_evidence(evidence):
        return {
            "status": "BLOCKED",
            "reason": "Invalid numerical values."
        }

    return {
        "status": "PASSED",
        "reason": "Evidence is valid and consistent."
    }


# Test 1: Valid evidence
gate_valid = unified_evidence_gate(valid_evidence)

# Test 2: Negative quantity
gate_negative = unified_evidence_gate(
    test_cases["Negative quantity"]
)

# Test 3: Mismatched product IDs
gate_mismatch = unified_evidence_gate(
    mismatched_evidence
)

# Test 4: Missing finance evidence
gate_missing = unified_evidence_gate({
    "inventory": valid_evidence["inventory"]
})

print("Valid evidence:", gate_valid["status"])
print("Negative quantity:", gate_negative["status"])
print("Mismatched products:", gate_mismatch["status"])
print("Missing finance:", gate_missing["status"])

assert gate_valid["status"] == "PASSED"
assert gate_negative["status"] == "BLOCKED"
assert gate_mismatch["status"] == "BLOCKED"
assert gate_missing["status"] == "BLOCKED"

print("\nUnified evidence gate tests: PASSED")


Valid evidence: PASSED
Negative quantity: BLOCKED
Mismatched products: BLOCKED
Missing finance: BLOCKED

Unified evidence gate tests: PASSED



### 22. Save Agent Evaluation Results


In [21]:

import json
from pathlib import Path
from datetime import datetime, timezone

# Create evaluation output folder
evaluation_dir = PROCESSED_DIR / "agent_evaluation"
evaluation_dir.mkdir(parents=True, exist_ok=True)

# Save individual question results
results_to_save = results_df.copy()

# Convert lists to JSON strings for CSV storage
for column in ["expected_tools", "selected_tools"]:
    results_to_save[column] = results_to_save[column].apply(
        json.dumps
    )

results_path = evaluation_dir / "nb7_tool_selection_results.csv"

results_to_save.to_csv(
    results_path,
    index=False
)

# Prepare evaluation summary
evaluation_summary = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "model": "llama3.2:latest",
    "allocation_strategy": "Strategy B - Exposure per Dollar",
    "questions_tested": int(len(results_df)),
    "strict_tool_selection_accuracy": float(
        results_df["exact_match"].mean()
    ),
    "average_initial_evidence_coverage": float(
        coverage_df["evidence_coverage"].mean()
    ),
    "initial_complete_cases": int(
        coverage_df["complete_evidence"].sum()
    ),
    "complete_cases_after_offline_recovery": int(
        recovery_df["evidence_complete"].sum()
    ),
    "numerically_valid_cases": int(
        values_df["valid_business_values"].sum()
    ),
    "unified_gate_tests_passed": all([
        gate_valid["status"] == "PASSED",
        gate_negative["status"] == "BLOCKED",
        gate_mismatch["status"] == "BLOCKED",
        gate_missing["status"] == "BLOCKED"
    ]),
    "real_purchases_executed": False
}

summary_path = evaluation_dir / "nb7_evaluation_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(evaluation_summary, f, indent=4)

# Verify saved files
assert results_path.exists()
assert summary_path.exists()

with open(summary_path, "r", encoding="utf-8") as f:
    saved_summary = json.load(f)

assert saved_summary == evaluation_summary

print("--- EVALUATION FILES ---")
print("Tool-selection results saved:", results_path.exists())
print("Evaluation summary saved:", summary_path.exists())

print("\n--- SAVED EVALUATION SUMMARY ---")
print("Questions tested:", saved_summary["questions_tested"])
print(
    "Strict tool-selection accuracy:",
    f"{saved_summary['strict_tool_selection_accuracy'] * 100:.2f}%"
)
print(
    "Initial evidence coverage:",
    f"{saved_summary['average_initial_evidence_coverage'] * 100:.2f}%"
)
print(
    "Complete after offline recovery:",
    saved_summary["complete_cases_after_offline_recovery"]
)
print(
    "Numerically valid cases:",
    saved_summary["numerically_valid_cases"]
)
print(
    "Unified gate tests passed:",
    saved_summary["unified_gate_tests_passed"]
)

print("\nEvaluation file verification: PASSED")


--- EVALUATION FILES ---
Tool-selection results saved: True
Evaluation summary saved: True

--- SAVED EVALUATION SUMMARY ---
Questions tested: 9
Strict tool-selection accuracy: 66.67%
Initial evidence coverage: 83.33%
Complete after offline recovery: 9
Numerically valid cases: 9
Unified gate tests passed: True

Evaluation file verification: PASSED



## Conclusion: End-to-End Testing and Evaluation

### Evaluation Objective

Evaluate the tool-selection performance, evidence completeness, numerical consistency, and basic safety controls of the hybrid Agentic AI retail decision-support system.

### Evaluation Results

The local Llama 3.2 model was evaluated using nine business questions across three products and three question categories.

| Evaluation Metric | Result |
|---|---:|
| Questions evaluated | 9 |
| Strict tool-selection accuracy | 66.67% |
| Average initial evidence coverage | 83.33% |
| Complete initial tool selection | 6/9 |
| Evidence complete after offline recovery | 9/9 |
| Numerically valid evaluation records | 9/9 |
| Unified evidence gate tests | Passed |

### Key Findings

1. Llama 3.2 correctly selected the expected tools for all tested inventory-only and finance-only questions.
2. The model failed strict tool-selection evaluation for all three combined questions, selecting only the inventory tool and omitting the finance tool.
3. A deterministic Python fallback successfully retrieved the missing financial records during offline evaluation.
4. Numerical validation rejected negative quantities, negative purchasing costs, and excess funded quantities.
5. Product consistency checks rejected evidence combining different product IDs.
6. Strategy B (Exposure per Dollar) was established as the evaluation reference after identifying that the saved business file contained allocations from both Strategies A and B.

### System Strengths

- Integrates demand forecasting, inventory planning, and financial risk analysis.
- Demonstrates real LLM-driven tool selection through Ollama.
- Uses a bounded tool-calling workflow.
- Provides deterministic evidence validation and recovery.
- Retains human approval requirements for purchasing decisions.
- Stores evaluation results for reproducibility.

### Limitations

- The evaluation contains only nine questions across three products.
- Initial tool selection is unreliable for multi-part questions.
- The recovery evaluation used saved datasets rather than exercising the complete agent for every question.
- Numerical consistency does not guarantee business recommendation quality.
- The inventory and financial environment relies on simulated assumptions.
- Forecasting errors may affect downstream decisions.
- No real purchasing transactions were performed.

### Overall Conclusion

The project demonstrates a functional hybrid Agentic AI prototype combining a local Llama 3.2 language model with deterministic business analysis tools and validation safeguards.

The evaluation highlights a clear tradeoff: LLM-based tool selection enables flexible interaction, while Python-based validation and evidence recovery improve completeness and numerical consistency.

The system is suitable as a research and portfolio prototype, but further testing and refinement are required before production deployment.

### Next Development Phase

Build a Streamlit application to demonstrate demand forecasts, inventory recommendations, budget analysis, and the LLM-based business assistant.
